# beat2body: run the 2D pipeline on Colab (README 4.3-4.7)

Runs the three pose tools (`pose_mediapipe.py`, `pose_dwpose.py`, `pose_openpose.py` in `2d/scripts/`)
on every video of the chosen categories, then runs the later steps of `pipeline.py` for each tool.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

Steps:
1. Install packages.
2. Choose categories and download their videos from the team repository.
3. Run the three pose tools.
4. Clean the keypoints, extract audio features, cut clips and split (README 4.4-4.7) for each tool.

Results:
- `2d/keypoints/<tool>/<category>/<video_id>.npz`: cleaned keypoints of each tool
- `2d/clips/<tool>/train|val|test/` and `2d/clips/<tool>/meta.csv`: clips of each tool
- `2d/audio/`, `2d/features/`: audio and audio features, shared by the three tools

Use `Runtime -> Run all`.

In [1]:
# 1. Check the GPU and install packages
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DWPose and OpenPose will run on CPU (slow)"
!pip -q install mediapipe onnxruntime-gpu
!pip -q install --no-deps easy-dwpose
# If pip reports that numpy was changed: Runtime -> Restart session, then continue from the next cell.

/bin/bash: line 1: nvidia-smi: command not found
No GPU: DWPose and OpenPose will run on CPU (slow)


In [5]:
from google.colab import drive
drive.mount("/content/drive")
# 2. Choose categories and download their videos
CATEGORIES = ["la_hiphop", "house", "middle_hiphop", "break", "pop", "lock", "waack", "krump", "street_jazz", "ballet_jazz"]
# All 10: la_hiphop, house, middle_hiphop, break, pop, lock, waack, krump, street_jazz, ballet_jazz
LIMIT = None      # videos per category for a quick test; None = all 21
REPO = "/content/beat2body"
SOURCE = "https://github.com/SAMMYBOOOOM/beat2body"
BRANCH = "main"

import os, glob, subprocess
if not os.path.exists(REPO):
    !git clone -q --depth 1 --filter=blob:none --sparse -b {BRANCH} {SOURCE} {REPO}
else:
    !cd {REPO} && git pull -q origin {BRANCH}
names = subprocess.run(["git", "ls-tree", "-r", "--name-only", "HEAD", "2d/raw_video"],
                       cwd=REPO, capture_output=True, text=True).stdout.splitlines()
VIDEOS = []
for c in CATEGORIES:
    vids = sorted(n[len("2d/raw_video/"):] for n in names if n.startswith(f"2d/raw_video/{c}/") and n.endswith(".mp4"))
    VIDEOS += vids[:LIMIT] if LIMIT else vids
patterns = " ".join(["'/*'", "'!/2d/raw_video/**/*.mp4'"] + [f"'/2d/raw_video/{v}'" for v in VIDEOS])
!cd {REPO} && git sparse-checkout set --no-cone {patterns}
for v in VIDEOS:
    print("OK     " if os.path.exists(f"{REPO}/2d/raw_video/{v}") else "MISSING", v)
print("scripts:", sorted(os.listdir(f"{REPO}/2d/scripts")))

# Pose outputs are copied to Google Drive while the tools run (every minute, in the background)
# and again at the end of cells 3a-3c, so after a disconnect the finished videos are restored here and skipped.
import shutil, threading, time
import numpy as np
SAVE = "/content/drive/MyDrive/beat2body_pose_raw"
def save_pose(tool, quiet=False):
    n = 0
    for f in glob.glob(f"{REPO}/2d/keypoints/*/*_{tool}.npz"):
        dst = f"{SAVE}/{os.path.basename(os.path.dirname(f))}/{os.path.basename(f)}"
        if os.path.exists(dst) and os.path.getmtime(dst) >= os.path.getmtime(f):
            continue
        try:
            np.load(f).files          # skip a file that is still being written
        except Exception:
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(f, dst)
        n += 1
    if not quiet or n:
        print(f"[{tool}] {n} new pose outputs saved to {SAVE}")
def autosave(every=60):
    while True:
        time.sleep(every)
        for tool in ["mediapipe", "dwpose", "openpose"]:
            try:
                save_pose(tool, quiet=True)
            except Exception as e:
                print("autosave:", e)
if not globals().get("AUTOSAVE_STARTED"):
    threading.Thread(target=autosave, daemon=True).start()
    AUTOSAVE_STARTED = True
restored = 0
for f in glob.glob(f"{SAVE}/*/*.npz"):
    dst = f"{REPO}/2d/keypoints/{os.path.basename(os.path.dirname(f))}/{os.path.basename(f)}"
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(f, dst)
        restored += 1
print(f"restored {restored} earlier pose outputs from {SAVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK      la_hiphop/gLH_sFM_c01_d16_mLH0_ch01.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH1_ch02.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH2_ch03.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH3_ch04.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH3_ch07.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH4_ch05.mp4
OK      la_hiphop/gLH_sFM_c01_d16_mLH5_ch06.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH0_ch08.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH0_ch14.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH1_ch09.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH2_ch10.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH3_ch11.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH4_ch12.mp4
OK      la_hiphop/gLH_sFM_c01_d17_mLH5_ch13.mp4
OK      la_hiphop/gLH_sFM_c01_d18_mLH0_ch15.mp4
OK      la_hiphop/gLH_sFM_c01_d18_mLH0_ch21.mp4
OK      la_hiphop/gLH_sFM_c01_d18_mLH1_ch16.mp4
OK      la_hiphop/gLH_sFM_c01_d18_mLH2_ch17.mp4
OK     

In [7]:
# Check that every category has 21 raw pose outputs per tool and no extra files
# (e.g. duplicates like "name (1).npz" created by Google Drive) before running cell 4.
K = f"{REPO}/2d/keypoints"
for c in sorted(os.listdir(K)):
    if os.path.isdir(f"{K}/{c}"):
        n = {t: len(glob.glob(f"{K}/{c}/*_{t}.npz")) for t in ["dwpose", "mediapipe", "openpose"]}
        extra = len(glob.glob(f"{K}/{c}/*.npz")) - sum(n.values())
        print(f"{c:15s} {n}  extra={extra}")

ballet_jazz     {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
break           {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
house           {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
krump           {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
la_hiphop       {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
lock            {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
middle_hiphop   {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
pop             {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
street_jazz     {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0
waack           {'dwpose': 21, 'mediapipe': 21, 'openpose': 21}  extra=0


In [ ]:
# 3a. MediaPipe (CPU)
# MediaPipe's GPU mode needs OpenGL, which Colab does not provide, so it runs on CPU here.
# Videos that already have output are skipped; add --overwrite to run them again.
%cd {REPO}
try:
    !python 2d/scripts/pose_mediapipe.py --device cpu
finally:
    save_pose("mediapipe")

In [ ]:
# 3b. DWPose (GPU)
%cd {REPO}
try:
    !python 2d/scripts/pose_dwpose.py
finally:
    save_pose("dwpose")

In [ ]:
# 3c. OpenPose (GPU, official CMU build)
# The first run compiles OpenPose (about 25 minutes) and saves the build to your Google Drive;
# later sessions restore it from there in a few minutes.
CACHE = "/content/drive/MyDrive/openpose_colab_build.tar.gz"
BIN = "/content/openpose/build/examples/openpose/openpose.bin"
if not os.path.exists(BIN) and os.path.exists(CACHE):
    print("restoring OpenPose from Google Drive")
    !tar xzf {CACHE} -C /content
    !apt-get -qq update > /dev/null && apt-get -qq install -y libgoogle-glog-dev libgflags-dev libprotobuf-dev libhdf5-serial-dev libatlas-base-dev libboost-all-dev libopencv-dev > /dev/null
!bash {REPO}/2d/scripts/colab_build_openpose.sh
if os.path.exists(BIN) and not os.path.exists(CACHE):
    print("saving the OpenPose build to Google Drive")
    !tar czf {CACHE} -C /content --exclude='*.o' --exclude='.git' openpose
%cd {REPO}
try:
    !python 2d/scripts/pose_openpose.py --backend official
finally:
    save_pose("openpose")

In [8]:
# 4. README 4.4-4.7 for each of the three tools, keeping only the cleaned keypoints:
#    keypoints/<tool>/<category>/<video_id>.npz   cleaned keypoints (4.4)
#    keypoints/<tool>/normalization.json
#    clips/<tool>/train|val|test/, clips/<tool>/meta.csv   (4.6 + 4.7)
#    audio/ and features/ (4.5) are shared by the three tools.
# The raw pose outputs (*_<tool>.npz) are deleted at the end.
import glob, os, shutil, subprocess, sys

def run(*args):
    r = subprocess.run([sys.executable, "2d/scripts/pipeline.py", *args], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()

TOOLS = ["dwpose", "mediapipe", "openpose"]
K, C = f"{REPO}/2d/keypoints", f"{REPO}/2d/clips"
CATS = sorted({v.split("/")[0] for v in VIDEOS})

run("audio", "--categories", *CATS)
for tool in TOOLS:
    raw = [f for c in CATS for f in glob.glob(f"{K}/{c}/*_{tool}.npz")]
    if not raw:
        print(f"[{tool}] no pose output, skipped")
        continue
    run("clean", "--tool", tool, "--categories", *CATS)
    run("clips", "--categories", *CATS)
    for c in CATS:
        os.makedirs(f"{K}/{tool}/{c}", exist_ok=True)
        for f in glob.glob(f"{K}/{c}/*.npz"):
            name = os.path.basename(f)
            if not any(name.endswith(f"_{t}.npz") for t in TOOLS):
                shutil.move(f, f"{K}/{tool}/{c}/{name}")
    shutil.move(f"{K}/normalization.json", f"{K}/{tool}/normalization.json")
    for split in ["train", "val", "test"]:
        dst = f"{C}/{tool}/{split}"
        shutil.rmtree(dst, ignore_errors=True)
        os.makedirs(dst)
        for f in glob.glob(f"{C}/{split}/*.npz"):
            shutil.move(f, dst)
    meta = open(f"{REPO}/2d/meta.csv").read().replace("2d/clips/", f"2d/clips/{tool}/")
    open(f"{C}/{tool}/meta.csv", "w").write(meta)
    os.remove(f"{REPO}/2d/meta.csv")
    for f in raw:
        os.remove(f)
    print(f"[{tool}] done -> 2d/keypoints/{tool}/ and 2d/clips/{tool}/")

p/gPO_sFM_c01_d12_mPO5_ch20: 886 frames, 60 beats (~129 BPM)
pop/gPO_sFM_c01_d12_mPO5_ch21: 886 frames, 60 beats (~129 BPM)
street_jazz/gJS_sFM_c01_d01_mJS0_ch01: 1439 frames, 126 beats (~164 BPM)
street_jazz/gJS_sFM_c01_d01_mJS1_ch02: 1280 frames, 60 beats (~90 BPM)
street_jazz/gJS_sFM_c01_d01_mJS1_ch07: 1280 frames, 60 beats (~90 BPM)
street_jazz/gJS_sFM_c01_d01_mJS2_ch03: 1152 frames, 63 beats (~100 BPM)
street_jazz/gJS_sFM_c01_d01_mJS3_ch04: 1048 frames, 59 beats (~112 BPM)
street_jazz/gJS_sFM_c01_d01_mJS4_ch05: 960 frames, 64 beats (~120 BPM)
street_jazz/gJS_sFM_c01_d01_mJS5_ch06: 886 frames, 59 beats (~129 BPM)
street_jazz/gJS_sFM_c01_d02_mJS0_ch08: 1439 frames, 126 beats (~164 BPM)
street_jazz/gJS_sFM_c01_d02_mJS1_ch02: 1280 frames, 60 beats (~90 BPM)
street_jazz/gJS_sFM_c01_d02_mJS1_ch11: 1280 frames, 60 beats (~90 BPM)
street_jazz/gJS_sFM_c01_d02_mJS2_ch03: 1152 frames, 63 beats (~100 BPM)
street_jazz/gJS_sFM_c01_d02_mJS3_ch04: 1048 frames, 59 beats (~112 BPM)
street_jazz/gJS_

In [9]:
# 5. Save the results to Google Drive
!mkdir -p /content/drive/MyDrive/beat2body_results && cp -r {REPO}/2d/keypoints {REPO}/2d/clips {REPO}/2d/features {REPO}/2d/categories.json /content/drive/MyDrive/beat2body_results/

In [10]:
# Wait until every file written to Google Drive has finished uploading.
drive.flush_and_unmount()

In [11]:
# 5b. Upload the results to Hugging Face (one dataset per tool, listed in the root README)
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
for tool in ["dwpose", "mediapipe", "openpose"]:
    if not os.path.isdir(f"{REPO}/2d/keypoints/{tool}"):
        print(f"[{tool}] no results, skipped")
        continue
    repo = f"SAMBOOM/AISTDance_{tool}"
    api.upload_folder(repo_id=repo, repo_type="dataset", folder_path=f"{REPO}/2d", path_in_repo="2d",
                      allow_patterns=[f"keypoints/{tool}/**", f"clips/{tool}/**", "features/**", "categories.json"],
                      commit_message=f"Upload {tool} results for {' '.join(CATEGORIES)}")
    print(f"[{tool}] uploaded to https://huggingface.co/datasets/{repo}")

[dwpose] uploaded to https://huggingface.co/datasets/SAMBOOM/AISTDance_dwpose
[mediapipe] uploaded to https://huggingface.co/datasets/SAMBOOM/AISTDance_mediapipe
[openpose] uploaded to https://huggingface.co/datasets/SAMBOOM/AISTDance_openpose


In [12]:
# Remove files on Hugging Face that are not in this run's results (stale clips from earlier uploads).
# Only run right after 5b in the same session; it stops if the local results are missing.
from huggingface_hub import CommitOperationDelete
for tool in ["dwpose", "mediapipe", "openpose"]:
    n_local = len(glob.glob(f"{REPO}/2d/clips/{tool}/*/*.npz"))
    if n_local < 3000:
        print(f"[{tool}] only {n_local} local clips, skipped (rerun cells 1, 2 and 4 first)")
        continue
    repo = f"SAMBOOM/AISTDance_{tool}"
    remote = api.list_repo_files(repo, repo_type="dataset")
    stale = [f for f in remote
             if f.startswith((f"2d/keypoints/{tool}/", f"2d/clips/{tool}/", "2d/features/"))
             and not os.path.exists(f"{REPO}/{f}")]
    print(f"[{tool}] {len(stale)} stale files")
    if stale:
        api.create_commit(repo, repo_type="dataset", operations=[CommitOperationDelete(f) for f in stale],
                          commit_message="Remove files from earlier uploads")

[dwpose] 132 stale files
[mediapipe] 132 stale files
[openpose] 132 stale files


In [13]:
# Summarize which files were removed from the openpose dataset.
from collections import Counter
print(Counter("/".join(f.split("/")[:4]) for f in stale))
print(Counter(os.path.basename(f).split("__")[0] for f in stale))
print(stale[:5])

Counter({'2d/clips/openpose/train': 53, '2d/clips/openpose/test': 41, '2d/clips/openpose/val': 38})
Counter({'la_hiphop': 132})
['2d/clips/openpose/test/la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000000.npz', '2d/clips/openpose/test/la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000060.npz', '2d/clips/openpose/test/la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000120.npz', '2d/clips/openpose/test/la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000180.npz', '2d/clips/openpose/test/la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000240.npz']


In [ ]:
### Results of the full run (10 categories, 210 videos)
- Pose outputs: 21 videos per category for each tool (DWPose, MediaPipe, OpenPose).
- Split by video: train 15 / val 3 / test 3 per category.
- Clips (240 frames, stride 60):

| tool | train | val | test |
|---|---|---|---|
| dwpose | 2200 | 454 | 430 |
| mediapipe | 2198 | 454 | 430 |
| openpose | 2200 | 454 | 430 |

Why MediaPipe has 2 fewer training clips: `pipeline.py clips` drops a clip when less than half of its joints were observed (`--min-valid 0.5`), because the rest would be filled in by interpolation (straight-line motion between observed points), which does not reflect the real dance. MediaPipe missed most joints in the first ~10 seconds of `ballet_jazz/gJB_sFM_c01_d08_mJB0_ch08`, so its clips starting at frames 0 and 60 were dropped. DWPose and OpenPose kept them. The cleaned keypoints of that video are still complete; only those two clip windows are missing. Val and test are identical for all three tools.

Normalization: keypoints are centered on the pelvis and scaled by the median torso length over all 210 videos (README 4.4), saved in `keypoints/<tool>/normalization.json`. Results from runs on fewer categories use a different scale and category index, so they should not be mixed with these.